In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.preprocessing import get_features

df_train = pd.read_parquet("data/processed/train.parquet")

X_train = df_train.drop(columns=["target"]).copy()
y_train = df_train["target"].copy()

numeric_features, categorical_features = get_features(df_train)


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)


max_iterations = 1000
tscv = TimeSeriesSplit(n_splits=3)

balanced_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=max_iterations,
                class_weight="balanced",
            ),
        ),
    ],
)


random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ],
)

lgbm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LGBMClassifier(is_unbalance=True, random_state=42, verbose=-1, n_jobs=-1),
        ),
    ],
)


pipelines = {
    "logistic_regression": balanced_pipeline,
    "random_forest": random_forest_pipeline,
    "lightgbm_unbalanced": lgbm_pipeline,
}


for name, pipeline in pipelines.items():
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring=["roc_auc", "average_precision"],
        n_jobs=-1,
    )

    print(f"=== {name} ===")
    print(f"Mean ROC-AUC:          {cv_results['test_roc_auc'].mean():.4f}")
    print(f"Mean Avg Precision:    {cv_results['test_average_precision'].mean():.4f}\n")


=== logistic_regression ===
Mean ROC-AUC:          0.7266
Mean Avg Precision:    0.4072

=== random_forest ===
Mean ROC-AUC:          0.7186
Mean Avg Precision:    0.3894

=== lightgbm_unbalanced ===
Mean ROC-AUC:          0.7323
Mean Avg Precision:    0.4159



# Understanding Week 6

## The Models

3 models were chosen. Logistic Regression (LR), which we have used previously, Random Forest (RF) and LightGBM (LGBM). These were chosen so we can analyze linear and non-linear model outputs. These are the results without tuning them.

| Score              | Logistic Regression | Random Forest | LightGBM |
| ------------------ | ------------------- | ------------- | -------- |
| Mean ROC-AUC       | 0.7266              | 0.7186        | 0.7323   |
| Mean Avg Precision | 0.4072              | 0.3894        | 0.4159   |

A notable finding is that RF scored lower than LR. You would think that non-linear would capture more complex patterns. Likely because RF is using default hyperparameters. The tuning step below tests this directly. LGBM beat both untuned.


In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

train_idx, val_idx = list(tscv.split(X_train))[-1]
tuning_cv = [(train_idx, val_idx)]

param_distributions = {
    "logistic_regression": {
        "classifier__C": uniform(0.01, 10),
    },
    "random_forest": {
        "classifier__n_estimators": randint(100, 500),
        "classifier__max_depth": randint(3, 20),
        "classifier__min_samples_leaf": randint(1, 50),
    },
    "lightgbm_unbalanced": {
        "classifier__num_leaves": randint(15, 127),
        "classifier__max_depth": randint(3, 12),
        "classifier__learning_rate": uniform(0.01, 0.2),
        "classifier__n_estimators": randint(100, 500),
    },
}

model_n_iter = {
    "logistic_regression": 15,
    "random_forest": 5,
    "lightgbm_unbalanced": 30,
}

tuned_pipelines = {}

for name, pipeline in pipelines.items():
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_distributions[name],
        n_iter=model_n_iter[name],
        scoring="roc_auc",
        cv=tuning_cv,
        random_state=42,
        n_jobs=1,
        verbose=2,
    )
    search.fit(X_train, y_train)

    print(f"=== {name} ===")
    print(f"Best params:        {search.best_params_}")
    print(f"Validation ROC-AUC: {search.best_score_:.4f}\n")

    tuned_pipelines[name] = search.best_estimator_

print("=== Tuned Pipelines ===")
for name, pipeline in tuned_pipelines.items():
    print(f"{name}: {pipeline}")

Fitting 1 folds for each of 15 candidates, totalling 15 fits
[CV] END ...................classifier__C=3.7554011884736247; total time=  30.2s
[CV] END .....................classifier__C=9.51714306409916; total time=  25.2s
[CV] END ...................classifier__C=7.3299394181140505; total time=  25.6s
[CV] END ....................classifier__C=5.996584841970366; total time=  24.7s
[CV] END ...................classifier__C=1.5701864044243652; total time=  25.6s
[CV] END ...................classifier__C=1.5699452033620265; total time=  24.5s
[CV] END ...................classifier__C=0.5908361216819946; total time=  24.0s
[CV] END ....................classifier__C=8.671761457749351; total time=  25.1s
[CV] END ....................classifier__C=6.021150117432088; total time=  26.1s
[CV] END ....................classifier__C=7.090725777960454; total time=  24.9s
[CV] END ..................classifier__C=0.21584494295802448; total time=  23.5s
[CV] END ....................classifier__C=9.709

## Tuning the models

To tune the models we used the last fold of the time series to see what are the most optimal hyperparameters. Hyperparameters are configuration variables set before training a machine learning model that control how the algorithm learn.

We only used the last fold for the search, not all 3 `TimeSeriesSplit` folds. Using all 3 would mean fitting every candidate 3 times, which is expensive since RF alone takes minutes per fit. Instead `RandomizedSearchCV` got the last fold's `(train_idx, val_idx)` pair as one fixed validation split: train on the earliest ~9/12 of the data, score on the most recent ~3/12. `val_idx` always comes after `train_idx` so this still respects time order, and it keeps the search cheap enough to actually run.

The search picks hyperparameters based on how well they score on that one slice. Reporting that same score as the final result would be misleading, since it's easy to land on a config that just happens to score well on the exact data it was judged on. So once the search finished we re-ran each winning pipeline through the full 3-fold `cross_validate`, the same one used for the untuned baseline. The numbers in the next section come from that run, not the "Validation ROC-AUC" printed during the search. Keeping those two separate is what makes the tuning honest instead of the search just grading itself.

There were issues with number of iterations. Since RF used much more time per iteration, so we first chose 5 iterations. After it ran we saw that both LR and LGBM could have ran much more iteration. We chose to have different iterables per model to test if LR and LGBM had reach it's celling.

| Model               | Iteration |
| ------------------- | --------- |
| Logistic Regression | 15        |
| Random Forest       | 5         |
| LightGBM            | 30        |


In [5]:
for name, pipeline in tuned_pipelines.items():
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=tscv,
        scoring=["roc_auc", "average_precision"],
        n_jobs=-1,
    )

    print(f"=== {name} (tuned) ===")
    print(f"Mean ROC-AUC:          {cv_results['test_roc_auc'].mean():.4f}")
    print(f"Mean Avg Precision:    {cv_results['test_average_precision'].mean():.4f}\n")


=== logistic_regression (tuned) ===
Mean ROC-AUC:          0.7267
Mean Avg Precision:    0.4072

=== random_forest (tuned) ===
Mean ROC-AUC:          0.7271
Mean Avg Precision:    0.4077

=== lightgbm_unbalanced (tuned) ===
Mean ROC-AUC:          0.7326
Mean Avg Precision:    0.4172



## Models After Tuning

After tuning the models we ran them again with their optimal hyperparameters. These were the scores of the models.

| Score              | Logistic Regression | Random Forest | LightGBM |
| ------------------ | ------------------- | ------------- | -------- |
| Mean ROC-AUC       | 0.7267              | 0.7271        | 0.7326   |
| Mean Avg Precision | 0.4072              | 0.4077        | 0.4172   |

After tuning RF beats LR. While LGBM barely moved after 6x more search. This means that LGBM was not under-search and was near it's celling. LR only moved 0.001, which can be attributed to noise. LightGBM > RF > LR, all three within 0.006 ROC-AUC.
